#### Deep Research


In [ ]:
import asyncio
import os
import sys
from pathlib import Path
from typing import Dict, Optional

import gradio as gr
import httpx
import requests
import sendgrid
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
from sendgrid.helpers.mail import Content, Email, Mail, To

from agents import (
    Agent,
    ModelSettings,
    OpenAIChatCompletionsModel,
    Runner,
    function_tool,
    gen_trace_id,
    handoff,
    set_tracing_disabled,
    trace,
)
from agents.extensions import handoff_filters

load_dotenv(override=True)

if os.environ.get("AGENTS_DISABLE_TRACING", "").strip().lower() in ("1", "true", "yes"):
    set_tracing_disabled(True)




### Model


In [ ]:
OPENROUTER_BASE_URL = os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o-mini")

_openrouter_client = AsyncOpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

llm_model = OpenAIChatCompletionsModel(
    model=OPENROUTER_MODEL,
    openai_client=_openrouter_client,
)

### Web search (SerperDev and DuckDuckGo fallback)


In [ ]:
# Web search usable with OpenRouter-only setups (no OpenAI hosted WebSearchTool).

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)


def _serper_search(query: str) -> Optional[str]:
    api_key = os.environ.get("SERPER_API_KEY")
    if not api_key:
        return None
    url = "https://google.serper.dev/search"
    payload = {"q": query, "num": 8}
    headers = {"X-API-KEY": api_key, "Content-Type": "application/json"}
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=20)
        response.raise_for_status()
        data = response.json()
        out = f"Search results for: {query}\n\n"
        if "organic" in data:
            for i, item in enumerate(data["organic"][:6], 1):
                out += f"{i}. {item.get('title', '')}\n"
                out += f"   {item.get('snippet', '')}\n"
                out += f"   {item.get('link', '')}\n\n"
        if "knowledgeGraph" in data and data["knowledgeGraph"].get("description"):
            out += f"\nKey info: {data['knowledgeGraph']['description']}\n"
        return out.strip() or None
    except Exception:
        return None


def _ddg_instant(query: str) -> str:
    r = httpx.get(
        "https://api.duckduckgo.com/",
        params={"q": query, "format": "json", "no_html": "1"},
        headers={"User-Agent": USER_AGENT},
        timeout=20.0,
    )
    r.raise_for_status()
    d = r.json()
    parts: list[str] = []
    if d.get("AbstractText"):
        parts.append(d["AbstractText"])
    for t in d.get("RelatedTopics", [])[:6]:
        if isinstance(t, dict) and t.get("Text"):
            parts.append(t["Text"])
    return "\n\n".join(parts) if parts else ""


def _ddg_html(query: str) -> str:
    r = httpx.get(
        "https://html.duckduckgo.com/html/",
        params={"q": query},
        headers={"User-Agent": USER_AGENT},
        timeout=25.0,
    )
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    lines: list[str] = []
    for a in soup.select("a.result__a")[:10]:
        title = a.get_text(strip=True)
        if title:
            lines.append(f"- {title}")
    if not lines:
        for s in soup.select(".result__snippet")[:6]:
            t = s.get_text(strip=True)
            if t:
                lines.append(t)
    return "\n".join(lines) if lines else ""


def run_web_search(query: str) -> str:
    """Return raw web snippets for the model to summarize."""
    s = _serper_search(query)
    if s:
        return s
    try:
        instant = _ddg_instant(query)
        if instant.strip():
            return instant
    except Exception:
        pass
    try:
        html = _ddg_html(query)
        if html.strip():
            return html
    except Exception as exc:
        return (
            f"Web search failed for {query!r}: {exc!s}. "
            "Set SERPER_API_KEY for reliable results, or retry with a shorter query."
        )
    return (
        f"No web results returned for {query!r}. "
        "Try a broader search term or set SERPER_API_KEY in .env."
    )


### Agents: clarifier, planner, writer, evaluator


In [ ]:
class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        description="Exactly 3 clarifying questions to narrow the research scope.",
    )


INSTRUCTIONS = (
    "You are a clarification agent. Given a research query, generate exactly 3 clarifying questions "
    "that narrow the scope. Cover: specific focus or angle, audience or use case, and depth, timeframe, "
    "or geography. Questions must be open-ended and actionable. Return exactly 3 questions."
)

clarifier_agent = Agent(
    name="ClarifierAgent",
    instructions=INSTRUCTIONS,
    model=llm_model,
    output_type=ClarifyingQuestions,
)

HOW_MANY_SEARCHES = 4

PLAN_INSTRUCTIONS = (
    f"You are a helpful research assistant. You receive: the user's query; the user's answers to clarifying questions; "
    f"and optional feedback from a prior evaluation asking for more coverage.\n"
    f"Propose {HOW_MANY_SEARCHES} web search items. Each item must include a precise search query and a short reason. "
    "Tune queries using clarifications (timeframe, region, audience, technical level). "
    "If refinement feedback is provided, bias new searches toward missing topics and suggested improvements. "
    "Avoid duplicating themes that prior searches already covered—explore complementary angles."
)


class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query.",
    )


planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLAN_INSTRUCTIONS,
    model=llm_model,
    output_type=WebSearchPlan,
)

INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, clarifications, and summarized web research.\n"
    "First outline the structure mentally, then produce the report as your final output.\n"
    "The final output must be in markdown, detailed, and at least 1000 words where evidence allows."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=llm_model,
    output_type=ReportData,
)

class EvaluationResult(BaseModel):
    score: int = Field(
        ge=1,
        le=10,
        description="Overall quality and completeness of the report vs the research query (1=very poor, 10=excellent).",
    )
    missing_topics: list[str] = Field(
        description="Important angles, subtopics, or evidence gaps that are missing or underdeveloped.",
    )
    suggested_improvements: str = Field(
        description="Concrete suggestions to strengthen the report (structure, depth, sourcing, balance).",
    )
    should_continue_research: bool = Field(
        description="True if more web research is needed before the report can be considered adequate.",
    )


INSTRUCTIONS = (
    "You are an expert research editor. You receive the original user research query (and any clarifications) "
    "and a draft research report in markdown.\n"
    "Critically assess coverage, depth, balance of perspectives, and use of evidence. "
    "List missing topics that warrant additional search. Suggest specific improvements.\n"
    "Set should_continue_research to true if score is below 7 or if major gaps remain that web research could fix. "
    "Set it to false if the report is already strong for the query."
)

evaluator_agent = Agent(
    name="EvaluatorAgent",
    instructions=INSTRUCTIONS,
    model=llm_model,
    output_type=EvaluationResult,
)


### Search agent


In [ ]:
@function_tool
def web_search(query: str) -> str:
    """Fetch raw web results for the given search query. Call once per planned search term."""
    return run_web_search(query)


INSTRUCTIONS = (
    "You are a research assistant. Each message includes: the user's original research intent, "
    "the clarifying questions and the user's answers (treat these as constraints), "
    "and a concrete search term with the reason it was chosen.\n"
    "Call web_search exactly once with that search term. Then synthesize the raw results into "
    "2–3 tight paragraphs (under 300 words). Capture main claims and caveats; skip fluff. "
    "No preamble or meta-commentary—summary body only."
)

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[web_search],
    model=llm_model,
    model_settings=ModelSettings(tool_choice="required"),
)


### Email agent (optional SendGrid)


In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send an email with the given subject and HTML body."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(os.environ.get("SENDGRID_FROM_EMAIL", "ed@edwarddonner.com"))
    to_email = To(os.environ.get("SENDGRID_TO_EMAIL", "ed.donner@gmail.com"))
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": str(response.status_code)}


INSTRUCTIONS = """You can send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. Use your tool once to send the report as clean HTML with a good subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model=llm_model,
)


### Research manager


In [ ]:
MAX_RESEARCH_ITERATIONS = 5
EVAL_PASS_THRESHOLD = 7

RECOMMENDED_PROMPT_PREFIX = (
    "# System context\nYou are part of a multi-agent system (OpenAI Agents SDK). "
    "Specialists are exposed as **tools** (agents-as-tools via Runner.run inside tool functions). "
    "Do not narrate internal tool calls to the end user.\n"
)

MANAGER_INSTRUCTIONS = (
    f"{RECOMMENDED_PROMPT_PREFIX}"
    "You are the Research Manager for one iteration of a deep-research job.\n"
    "Tools:\n"
    "- `run_planner`: Pass a rich planning_context string (query, clarifications, prior evaluator JSON, "
    "notes on what to avoid duplicating). Returns a JSON search plan.\n"
    "- `run_search_item`: Call once per planned query with that item's query and reason.\n"
    "- `run_searches_from_plan_tool`: After planning, run all planned searches in parallel.\n"
    "- `run_writer`: After searches, build the structured report from accumulated snippets.\n"
    "- `run_evaluator`: Pass the latest markdown report string.\n"
    "Standard flow: run_planner → run_searches_from_plan_tool (or individual run_search_item) → "
    "run_writer → run_evaluator.\n"
    f"When evaluator score ≥ {EVAL_PASS_THRESHOLD} and should_continue_research is false, "
    "you are done with this iteration.\n"
    "End with structured output `ManagerIterationResult` (latest_evaluator_score from the evaluator tool, "
    "needs_more_search aligned with should_continue_research, stop_research when done)."
)


class ManagerIterationResult(BaseModel):
    iteration_notes: str = Field(description="Brief status for logging.")
    latest_evaluator_score: int = Field(
        ge=0,
        le=10,
        description="Last evaluator score; 0 if not run.",
    )
    needs_more_search: bool = Field(
        description="True if another outer research iteration should run.",
    )
    stop_research: bool = Field(description="True if the outer loop should stop after this round.")


def _normalize_three_questions(raw: list[str]) -> list[str]:
    cleaned = [q.strip() for q in raw if q and str(q).strip()]
    out = cleaned[:3]
    while len(out) < 3:
        out.append("Any other constraints or preferences for this research?")
    return out


class ResearchManager:
    async def run(self, query: str, clarification_answers: str | None = None):
        """Clarifying questions first; then autonomous manager iterations with evaluator-driven replanning."""
        trace_id = gen_trace_id()
        trace_line = f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}"
        with trace("Research trace", trace_id=trace_id):
            answers = (clarification_answers or "").strip()
            if not answers:
                try:
                    cq = await Runner.run(
                        clarifier_agent,
                        f"Research query:\n{(query or '').strip() or '(empty — ask generic scoping questions)'}",
                    )
                    parsed = cq.final_output_as(ClarifyingQuestions)
                    qs = _normalize_three_questions(parsed.questions)
                except Exception as exc:
                    yield (
                        f"{trace_line}\n\n"
                        "### Clarifying questions\n\n"
                        f"_Could not generate questions ({exc!s}). Add details in “Your answers” and run again._"
                    )
                    return

                body = "\n".join(f"{i + 1}. {q}" for i, q in enumerate(qs))
                yield (
                    f"{trace_line}\n\n"
                    "### Clarifying questions\n\n"
                    f"{body}\n\n"
                    "---\n\n"
                    "**Answer these in “Your answers”**, then click **Run research**."
                )
                return

            ctx: dict = {
                "query": query,
                "clarifications": answers,
                "last_plan": None,
                "search_results": [],
                "last_report": None,
                "last_eval_json": None,
            }

            yield f"{trace_line}\n\nStarting autonomous research…"

            for round_i in range(1, MAX_RESEARCH_ITERATIONS + 1):
                yield f"**Iteration {round_i}/{MAX_RESEARCH_ITERATIONS}** — manager planning / searching / writing / evaluating…"

                @function_tool
                async def run_planner_tool(planning_context: str) -> str:
                    """Create or refine a WebSearchPlan."""
                    result = await Runner.run(planner_agent, planning_context)
                    plan = result.final_output_as(WebSearchPlan)
                    ctx["last_plan"] = plan
                    return plan.model_dump_json()

                @function_tool
                async def run_search_item_tool(search_term: str, reason: str) -> str:
                    """Run one web search; results accumulate for the writer."""
                    block = (
                        f"Original research query:\n{ctx['query']}\n\n"
                        f"Clarifying questions and user answers:\n{ctx['clarifications']}\n\n"
                        f"Search term: {search_term}\nReason for searching: {reason}"
                    )
                    result = await Runner.run(search_agent, block)
                    out = str(result.final_output)
                    ctx["search_results"].append(f"---\nTerm: {search_term}\n{out}")
                    return out

                @function_tool
                async def run_searches_from_plan_tool() -> str:
                    """Execute all items in the last plan in parallel."""
                    plan: WebSearchPlan | None = ctx.get("last_plan")
                    if not plan or not plan.searches:
                        return "No plan available; call run_planner first."

                    async def one(item):
                        block = (
                            f"Original research query:\n{ctx['query']}\n\n"
                            f"Clarifying questions and user answers:\n{ctx['clarifications']}\n\n"
                            f"Search term: {item.query}\nReason for searching: {item.reason}"
                        )
                        try:
                            result = await Runner.run(search_agent, block)
                            out = str(result.final_output)
                            ctx["search_results"].append(f"---\nTerm: {item.query}\n{out}")
                            return f"OK: {item.query}"
                        except Exception as exc:
                            return f"Failed: {item.query} ({exc})"

                    results = await asyncio.gather(*[one(s) for s in plan.searches])
                    return "\n".join(results)

                @function_tool
                async def run_writer_tool() -> str:
                    """Synthesize accumulated search snippets into the structured report."""
                    bundle = "\n\n".join(ctx["search_results"]) if ctx["search_results"] else "(no search results yet)"
                    writer_input = (
                        f"Original query:\n{ctx['query']}\n\n"
                        f"Clarifications (Q&A):\n{ctx['clarifications']}\n\n"
                        f"Summarized search results:\n{bundle}"
                    )
                    result = await Runner.run(writer_agent, writer_input)
                    report = result.final_output_as(ReportData)
                    ctx["last_report"] = report
                    return report.markdown_report

                @function_tool
                async def run_evaluator_tool(report_markdown: str) -> str:
                    """Score the report; JSON is stored for the next iteration's planner."""
                    ev_input = (
                        f"Original query:\n{ctx['query']}\n\n"
                        f"User clarifications:\n{ctx['clarifications']}\n\n"
                        f"Report to evaluate (markdown):\n{report_markdown}"
                    )
                    result = await Runner.run(evaluator_agent, ev_input)
                    ev = result.final_output_as(EvaluationResult)
                    ctx["last_eval_json"] = ev.model_dump_json()
                    return ctx["last_eval_json"]

                refinement = ctx["last_eval_json"] or "None yet (first iteration)."
                user_msg = (
                    f"--- Research iteration {round_i} of {MAX_RESEARCH_ITERATIONS} ---\n"
                    f"Original query:\n{query}\n\n"
                    f"User clarifications (answers):\n{answers}\n\n"
                    f"Prior evaluator JSON:\n{refinement}\n\n"
                    f"Accumulated search snippets: {len(ctx['search_results'])}.\n\n"
                    "For run_planner, pass one planning_context string that includes the query, clarifications, "
                    "the prior evaluator JSON, and instructions to avoid duplicate themes when refining.\n"
                    "Prefer run_searches_from_plan_tool after planning to run all planned searches in one step.\n"
                    "Then run_writer_tool, run_evaluator_tool with that markdown.\n"
                    f"If score ≥ {EVAL_PASS_THRESHOLD} and should_continue_research is false, set stop_research true.\n"
                    "End with ManagerIterationResult."
                )

                manager_agent = Agent(
                    name="ResearchManagerAgent",
                    instructions=MANAGER_INSTRUCTIONS,
                    model=llm_model,
                    tools=[
                        run_planner_tool,
                        run_search_item_tool,
                        run_searches_from_plan_tool,
                        run_writer_tool,
                        run_evaluator_tool,
                    ],
                    output_type=ManagerIterationResult,
                )

                await Runner.run(manager_agent, user_msg, max_turns=45)

                last_score = 0
                needs_more = True
                if ctx["last_eval_json"]:
                    ev = EvaluationResult.model_validate_json(ctx["last_eval_json"])
                    last_score = ev.score
                    needs_more = ev.should_continue_research

                yield (
                    f"Round {round_i}: evaluator score={last_score}, "
                    f"should_continue_research={needs_more}, snippets={len(ctx['search_results'])}"
                )

                if last_score >= EVAL_PASS_THRESHOLD and not needs_more:
                    break

            report = ctx.get("last_report")
            if report is None:
                yield "Research ended without a report."
                return

            yield "### Final report\n\n" + report.markdown_report

            if os.environ.get("SENDGRID_API_KEY"):
                yield "_Sending email (SendGrid)…_"
                try:
                    await Runner.run(email_agent, report.markdown_report, max_turns=8)
                    yield "_Email step completed._"
                except Exception as exc:
                    yield f"_Email step failed: {exc!s}_"
            else:
                yield "_Skipping email (set SENDGRID_API_KEY to enable)._"


In [ ]:
email_handoff = handoff(
    agent=email_agent,
    tool_description_override="Transfer to the email agent to send the finalized HTML report.",
    input_filter=handoff_filters.remove_all_tools,
)
email_handoff


In [ ]:
planner_tool = planner_agent.as_tool(
    tool_name="plan_searches",
    tool_description="Plan web searches given query, clarifications, and optional evaluator feedback.",
)
search_tool = search_agent.as_tool(
    tool_name="run_web_search",
    tool_description="Run one web search and return a short summary for the report.",
)
writer_tool = writer_agent.as_tool(
    tool_name="write_report",
    tool_description="Write the structured markdown report from collected search summaries.",
)
eval_tool = evaluator_agent.as_tool(
    tool_name="evaluate_report",
    tool_description="Score the report and list gaps; may request more research.",
)
len([planner_tool, search_tool, writer_tool, eval_tool])


### Gradio UI


In [ ]:
async def get_clarifying_questions(query: str) -> str:
    if not query.strip():
        return "Enter a research topic first."
    out = ""
    async for chunk in ResearchManager().run(query, None):
        out = chunk
    return out


async def run_research_stream(query: str, answers: str):
    if not query.strip():
        yield "Enter your research topic."
        return
    if not answers.strip():
        yield "Use **Get clarifying questions**, answer them in **Your answers**, then run again."
        return
    accumulated = ""
    async for chunk in ResearchManager().run(query, answers):
        accumulated += chunk + "\n\n"
        yield accumulated


with gr.Blocks(theme=gr.themes.Default(primary_hue="sky")) as ui:
    gr.Markdown(
        "# Deep Research\n\n"
        "1. Enter a topic and click **Get clarifying questions**.\n"
        "2. Answer all three in **Your answers**.\n"
        "3. Click **Run research** — progress streams below; the final report appears at the end.\n\n"
        "_Uses OpenRouter (`OPENROUTER_API_KEY`). Optional: `SERPER_API_KEY` for better web search; "
        "`SENDGRID_API_KEY` to email the report._"
    )
    query_textbox = gr.Textbox(
        label="Research topic",
        placeholder="e.g. Impact of small modular reactors on grid reliability in the EU",
        lines=2,
    )
    with gr.Row():
        questions_btn = gr.Button("Get clarifying questions", variant="secondary")
        run_btn = gr.Button("Run research", variant="primary")

    questions_display = gr.Markdown()
    answers_box = gr.Textbox(
        label="Your answers",
        placeholder="1) ... 2) ... 3) ...",
        lines=5,
    )
    progress = gr.Markdown()

    questions_btn.click(fn=get_clarifying_questions, inputs=query_textbox, outputs=questions_display)
    run_btn.click(fn=run_research_stream, inputs=[query_textbox, answers_box], outputs=progress)
    query_textbox.submit(fn=get_clarifying_questions, inputs=query_textbox, outputs=questions_display)


### 9. Launch


In [ ]:
ui.launch(inbrowser=True)
